# The complete API: from Tool to Eval · 2.1.9

Input subtotals 100 and 200 → Context, Tool, Skill, Agent, System decision and Environment → accumulated total 348. Source is visible here; Python without inference.

In [ ]:
"""A small, offline invoice: Tool → Agent → System → Environment, with Eval."""
import pydantic
import agentic_systems as toolkit

class Invoice(pydantic.BaseModel):
    subtotal: int
    total: int | None = None
    currency: str | None = None
    status: str | None = None

@toolkit.tool
def quote(subtotal: int, *, context: toolkit.Context) -> dict:
    """Calculate integer money using the domain's tax policy."""
    return {'total': subtotal * (100 + context.data['tax_percent']) // 100, 'currency': context.data['currency']}
domain = toolkit.Context({'tax_percent': 16, 'currency': 'MXN'}, sources=('tax-policy-v1',))
capability = toolkit.skill(name='invoicing', tools=(quote,), instructions='Amounts are integer MXN.')
memory = toolkit.Memory(capacity=3)
agent_graph = toolkit.Graph(Invoice)
agent_graph.add_node('quote', quote)
agent_graph.add_edge(toolkit.START, 'quote')
agent_graph.add_edge('quote', toolkit.END)
agent = toolkit.Agent('accountant', graph=agent_graph, skills=(capability,), context=domain, memory=memory)
system_graph = toolkit.Graph(Invoice)
system_graph.add_node('accountant', agent)
def approve_step(state):
    return {'status': 'approved'}
system_graph.add_node('approve', approve_step)
def review_step(state):
    return {'status': 'review'}
system_graph.add_node('review', review_step)
system_graph.add_edge(toolkit.START, 'accountant')
system_graph.add_conditional_edges('accountant', lambda state: 'review' if state.total > 1000 else 'approve', ['approve', 'review'])
system_graph.add_edge('approve', toolkit.END)
system_graph.add_edge('review', toolkit.END)
system = toolkit.System('billing', graph=system_graph, context=domain)
environment_graph = toolkit.Graph(Invoice)
environment_graph.add_node('billing', system)
environment_graph.add_edge(toolkit.START, 'billing')
environment_graph.add_edge('billing', toolkit.END)
environment = toolkit.Environment('batch', graph=environment_graph, records=({'subtotal': 100}, {'subtotal': 200}), state={'sum': 0}, update=lambda state, record, result: {'sum': state['sum'] + result.data['total']})
evaluation = toolkit.Eval('invoice-contract', system, cases=({'input': {'subtotal': 100}, 'expected': {'ok': True}},), checks=(lambda result, case: {'ok': result.data['total'] == 116 and result.data['currency'] == 'MXN'},))


In [ ]:
single = agent.run({'subtotal': 100})
team = system.run({'subtotal': 100})
batch = environment.run()
checked = evaluation.run()
assert single.ok and team.ok and batch.ok and checked.ok
assert batch.data['summary']['state'] == {'sum': 348}
print(single.data, team.data, batch.data['summary'])
print(batch.lineage().render())

print(single.human_results())


Load a local Skill only from a trusted Python source folder.

In [ ]:
from pathlib import Path
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'examples/common_graph/skills/pricing').is_dir()))
loaded = toolkit.load_skill(ROOT / 'examples/common_graph/skills/pricing')
assert loaded.tools
print(loaded.inspect())


A node function explicitly projects input and invokes the Tool. Evidence is retained automatically.

In [ ]:
class Nested(pydantic.BaseModel):
    order: Invoice
    total: int | None = None
    currency: str | None = None
projection = toolkit.Graph(Nested)
def quote_step(state):
    return quote.run(state.order.model_dump(mode="json", exclude_unset=True))
projection.add_node('quote', quote_step)
projection.add_edge(toolkit.START, 'quote')
projection.add_edge('quote', toolkit.END)
projected = toolkit.Agent('nested', graph=projection, context=domain).run({'order': {'subtotal': 100}})
assert projected.ok and projected.data['total'] == 116
print(projected.data)


runtime and framework select an atomic actor; LangGraph still coordinates execution.

In [ ]:
operator = toolkit.Agent('operator', runtime=toolkit.runtime(provider='python-runtime'), framework=toolkit.framework('native'), skills=(capability,), context=domain)
operated = operator.run({'subtotal': 100}, timeout_s=10)
assert operated.ok and operated.data['total'] == 116
restored = toolkit.RunResult(**operated.to_dict())
assert restored.check_invariants().ok
print(restored.data)


Pause and resume inside the graph, within the same process and program.

In [ ]:
approval = toolkit.Graph(Invoice)
def approval_step(state):
    return {'status': 'approved' if toolkit.pause({'question': 'Approve?'}) else 'rejected'}
approval.add_node('approval', approval_step)
approval.add_edge(toolkit.START, 'approval')
approval.add_edge('approval', toolkit.END)
program = toolkit.System('approval', graph=approval).compile()
waiting = program.run({'subtotal': 100})
assert waiting.status == 'paused'
resumed = program.resume(waiting, True)
assert resumed.ok and resumed.data['status'] == 'approved'
print(resumed.data)


Serving resources: declarations do not start processes, download weights or validate GPU availability.

In [ ]:
artifact = toolkit.model_artifact('Qwen/Qwen3-0.6B', revision='main', metadata={'scope': 'declaration only; resolved weights pending'})
server = toolkit.model_server(artifact, max_model_len=2048, max_num_seqs=1, gpu_memory_utilization=0.75)
print(artifact.inspect())
print(server.inspect())
